## This demo showcases the implementation of user story 589

This notebook shows the following features:
- Precise implementation of the staging endpoints: inputs and outputs of each endpoint of the staging must be compliant according to ogc standards
- Ability to stage a STAC ItemCollection from a single link

In [ ]:
import requests
import os
import pprint
import time
import pystac
# Init environment before running a demo notebook.
from resources.utils import *

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)
session = requests.Session()
auxip_client, cadip_client, catalog_client, staging_client, prip_client = init_demo()

if os.getenv("RSPY_LOCAL_MODE") == "1":
    href_cadip = "http://rs-server-cadip:8000"
    href_adgs = "http://rs-server-adgs:8000"
else:
    href_cadip = href_adgs = os.environ["RSPY_WEBSITE"]
    session.cookies.set("session", os.environ["RSPY_OAUTH2_COOKIE"])

cadip_collection_id = "cadip_sentinel1"
adgs_collection_id = "adgs"
TIMEOUT = 10

In [ ]:
# Init the dask cluster
from resources.dask_utils import *
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.dask_utils import *

# Use the staging cluster
dask_gateway = dask_gateway_staging
dask_cluster = dask_cluster_staging

In [ ]:
# Create a test collection 
collection = create_test_collection()

### Check the newly created collection with the rs-server-catalog. It should be empty.

In [ ]:
# Check the catalog for my_test_collection
collection = catalog_client.get_collection(TEST_COLLECTION)
for item in collection.get_items():
    print(f"Item {item.id} has {len(item.assets)} assets")

### Display ogc compliant information of processes

In [ ]:
# Existing process
process = staging_client.get_process("staging")
print(f"Staging process information: {process}")

# Unexisting process
unexisting_process = staging_client.get_process("unexisting_process")
print(f"Unexisting process information: {unexisting_process}")

### Check the jobs table

In [ ]:
jobs = staging_client.get_jobs()
if jobs.get("numberMatched") > 0:
    delete_jobs = True
    if cluster_mode == True:
        delete_jobs = input(f"There are {jobs.get('numberMatched')} jobs in the table. Do you want to delete them all (y/n)?").lower().strip() == 'y'
    if delete_jobs:
        print("Deleting all the jobs...")
        for job in jobs.get("jobs"):
            delete_response = staging_client.delete_job(job.get('jobID'))
        # Check that the jobs have been deleted
        jobs = staging_client.get_jobs()
        print(f"Existing jobs: {jobs}")

### Stage one item from Cadip station and one item from adgs station using valid links
Here we provide valid links with items that exist → the staging process is expected to succeed in both cases.

In [ ]:
# Current examples
#http://rs-server-cadip:8000/cadip/search?ids=S1A_20231120061537234567&collections=cadip_sentinel1
#http://rs-server-adgs:8000/auxip/search?ids=S1A_OPER_AUX_PREORB_OPOD_20240527T062732_V20240527T062732_20240527T062732.EOF&collections=adgs
staging_resp_list = []
staging_info = {
    "station": ["cadip", "auxip"],
    "id": ["S1A_20231120061537234567", "S1A_OPER_AUX_PREORB_OPOD_20240527T062732_V20240527T062732_20240527T062732.EOF"],
    "collection": ["cadip_sentinel1", "adgs"],
    "href": [href_cadip, href_adgs]
}
for i in range(len(staging_info["station"])):
    staging_link = f"{staging_info['href'][i]}/{staging_info['station'][i]}/search?ids={staging_info['id'][i]}&collections={staging_info['collection'][i]}"
    print(f"Launch staging with the following link: {staging_link}")
    staging_resp_list.append(staging_client.run_staging(staging_link, TEST_COLLECTION))

In [ ]:
started_job_id_list = []

for resp in staging_resp_list:
    staging_client.wait_for_jobs(resp, logger)

### Staging fails if the input link is not an url

In [ ]:
from rs_client.ogcapi.ogcapi_client import OgcValidationException

# On client side, we check if the input link is an url
staging_link = "this_is_not_a_link"
try:
    staging_client.run_staging(staging_link, TEST_COLLECTION)
except OgcValidationException as e:
    print(f"Staging failed with error: {e}")

### Stage one item from Cadip station and one item from adgs station using unvalid links

- For cadip provide an invalid domain name in the link which doesn't correspond to one of the existing cadip or adgs servers ("http://rs-server-cadi:8000" with "p" missing in the domain name)
- For adgs staging, give a link with a valid domain name but with an item identifier that doesn't exist (item  S1A_OPER_AUX_PREORB_OPOD_20240527T062732_V20240527T062732_20240527T062733.EOF doesn't exist)

→ The staging process is expected to fail in both cases

In [ ]:
# Current examples
#- missing "p" in the domain name -> http://rs-server-cadi:8000/cadip/search?ids=S1A_20231120061537234567&collections=cadip_sentinel1
#- Unexisting item identifier -> http://rs-server-adgs:8000/auxip/search?ids=S1A_OPER_AUX_PREORB_OPOD_20240527T062732_V20240527T062732_20240527T062733.EOF&collections=adgs
staging_resp_list = []
staging_info = {
    "station": ["cadip", "auxip"],
    "id": ["S1A_20231120061537234567", "S1A_OPER_AUX_PREORB_OPOD_20240527T062732_V20240527T062732_20240527T062733.EOF"], # Here S1A_OPER_AUX_PREORB_OPOD_20240527T062732_V20240527T062732_20240527T062733.EOF
    "collection": ["cadip_sentinel1", "adgs"],
    "href": ["http://rs-server-cadi:8000", href_adgs] # Link with an unvalid domain name, here there is a missing "p" in the domain name
}
for i in range(len(staging_info["station"])):
    staging_link = f"{staging_info['href'][i]}/{staging_info['station'][i]}/search?ids={staging_info['id'][i]}&collections={staging_info['collection'][i]}"
    print(f"Launch staging with the following link: {staging_link}")
    staging_resp_list.append(staging_client.run_staging(staging_link, TEST_COLLECTION))

In [ ]:
for resp in staging_resp_list:
    try:
        staging_client.wait_for_jobs(resp, logger)
    except RuntimeError:
        pass

### Check that the information of a job that doesn't exist is an error with an ogc compliant format

In [ ]:
unexisting_job_id = "12345-6789"
job_info = staging_client.get_job_info(unexisting_job_id)
print(job_info)
assert "type" in job_info
assert job_info["detail"] == 'Job with ID 12345-6789 not found'

### Check the catalog for my_test_collection. Ten items should be present now (one from CADIP station and nine from AUXIP station)

In [ ]:
# Check the catalog for my_test_collection
result = list(catalog_client.get_collection(TEST_COLLECTION).get_items())
print (f"{len(result)} items before removing")
for item in result:
    print(f"Item {item.id} has {len(item.assets)} assets")
assert len(result) == 2

### Check the jobs results

In [ ]:
# Check that each of the job previously launched are successful
for job_id in started_job_id_list:
    job_results = staging_client.get_job_results(job_id)
    print(f"Results from job {job_id}: {job_results}")
    assert job_results == "successful"

### Check that the result of a job that doesn't exist is an error with an ogc compliant format

In [ ]:
unexisting_job_id = "12345-6789"
job_results = staging_client.get_job_results(unexisting_job_id)
print(f"Results from an unexisting job {unexisting_job_id}: {job_results}")

assert "type" in job_results
assert job_results["detail"] == 'Job with ID 12345-6789 not found'

### Delete the whole collection

In [ ]:
result = catalog_client.remove_collection(TEST_COLLECTION)
assert result.json()["deleted collection"] == TEST_COLLECTION
pp.pprint(result.json())

In [ ]:
shutdown = False
if shutdown:
    # You can scale the clusters to 0 workers
    dask_gateway.scale_cluster(dask_cluster.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway, dask_cluster.name)

    # Close the python objects
    close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.